# LoRA Trainer — Fern Dataset with CLIPScore Evaluation

This notebook runs SD1.5 LoRA training on the **fern_new** dataset (28 images, 1024×1024, tag-style captions)
and evaluates training effectiveness using **CLIPScore** (CLIP image-to-image similarity between generated results
and the training dataset centroid).

**Runtime:** Select `GPU → T4` in Colab before running.

## 1. Setup

In [ ]:
# Clone repo and install
!git clone https://github.com/neverbiasu/lora-trainer.git /content/lora-trainer
%cd /content/lora-trainer
!pip install -e '.[dev]' -q

In [ ]:
# Verify GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 2. Upload Dataset

Upload `fern_new.zip` to `/content/` using the file browser on the left, or run the cell below to use the upload dialog.

In [ ]:
from pathlib import Path

# Option A: Upload via dialog
# from google.colab import files
# uploaded = files.upload()  # select fern_new.zip
# !mv fern_new.zip /content/fern_new.zip

# Option B: Google Drive (uncomment if dataset is on Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/fern_new.zip /content/fern_new.zip

# Extract
DATASET_ZIP = Path('/content/fern_new.zip')
DATASET_DIR = Path('/content/fern_dataset')

if DATASET_ZIP.exists():
    !unzip -o {DATASET_ZIP} -d {DATASET_DIR}
    # Handle nested directory: if zip extracts into a subfolder, flatten
    subdirs = [d for d in DATASET_DIR.iterdir() if d.is_dir()]
    if len(subdirs) == 1 and not any(DATASET_DIR.glob('*.png')):
        nested = subdirs[0]
        print(f'Flattening nested dir: {nested.name}')
        for f in nested.iterdir():
            f.rename(DATASET_DIR / f.name)
        nested.rmdir()
    
    imgs = list(DATASET_DIR.glob('*.png')) + list(DATASET_DIR.glob('*.jpg'))
    txts = list(DATASET_DIR.glob('*.txt'))
    print(f'Dataset: {len(imgs)} images, {len(txts)} captions')
else:
    print(f'ERROR: {DATASET_ZIP} not found. Upload it first!')

## 3. Configure Training

In [ ]:
import yaml

config = {
    'model': {'base_model': 'sd15'},
    'data': {
        'dataset_path': str(DATASET_DIR),
        'resolution': 512,
        'cache_latents': False,
    },
    'lora': {
        'rank': 32,
        'alpha': 32,
        'apply_text_encoder': True,
    },
    'training': {
        'learning_rate': 1e-4,
        'text_encoder_lr': 5e-5,
        'lr_scheduler': 'constant_with_warmup',
        'lr_warmup_steps': 100,
        'min_snr_gamma': 5.0,
        'batch_size': 1,
        'max_train_steps': 1000,
        'mixed_precision': 'fp16',
        'seed': 42,
        'save_every_n_steps': 500,
    },
    'sampling': {
        'prompts': [
            'fern, purple hair, anime girl, portrait, elf',
            'fern, purple hair, elf mage, fantasy, serious expression',
            'a girl with long purple hair, anime, illustration',
        ],
        'seeds': [42, 123, 999],
        'num_inference_steps': 20,
        'guidance_scale': 7.5,
        'sample_every_n_steps': 500,
    },
    'validation': {
        'assert_effective_training': True,
        'max_loss_ratio': 1.5,
        'min_lora_delta_l2': 1e-6,
        'min_pixel_mae': 0.02,
        'min_delta_clip': 0.005,
    },
    'output': {'output_dir': '/content/runs/fern_clipscore'},
}

CONFIG_PATH = Path('/content/config_fern.yaml')
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(f'Config saved to {CONFIG_PATH}')
print(yaml.safe_dump(config, sort_keys=False))

## 4. Run Training

This runs the full pipeline: baseline sampling → training (1000 steps) → final sampling → CLIPScore evaluation.

In [ ]:
%cd /content/lora-trainer

# Dry run first to validate config
!python -m src.lora_trainer.cli \
    --config {CONFIG_PATH} \
    --dataset {DATASET_DIR} \
    --dry-run

In [ ]:
# Full training run
!python -m src.lora_trainer.cli \
    --config {CONFIG_PATH} \
    --dataset {DATASET_DIR}

## 5. Inspect Results

In [ ]:
import json
from pathlib import Path

# Find latest run
runs_dir = Path('/content/runs/fern_clipscore')
run_dirs = sorted([d for d in runs_dir.iterdir() if d.is_dir() and d.name.startswith('run_')])
run_path = run_dirs[-1]
print(f'Latest run: {run_path}')

# Load metadata
meta_path = run_path / 'metadata.json'
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    metrics = meta.get('training_metrics', {})
    print(f"\n=== Training Metrics ===")
    print(f"Steps:          {metrics.get('total_steps')}")
    print(f"First loss:     {metrics.get('first_loss')}")
    print(f"Final loss:     {metrics.get('final_loss')}")
    print(f"Loss ratio:     {metrics.get('loss_ratio')}")
    print(f"LoRA delta L2:  {metrics.get('lora_delta_l2')}")
    print(f"\n=== CLIPScore Evaluation ===")
    print(f"Baseline CLIP:  {metrics.get('baseline_clip_sim')}")
    print(f"LoRA CLIP:      {metrics.get('lora_clip_sim')}")
    print(f"Delta CLIP:     {metrics.get('delta_clip')}")
    print(f"Pixel MAE:      {metrics.get('mean_pixel_mae')}")
    print(f"\n=== Effectiveness ===")
    print(f"Passed:         {metrics.get('effectiveness_passed')}")
    print(f"Reasons:        {metrics.get('effectiveness_reasons')}")
else:
    print('No metadata.json found')

In [ ]:
from IPython.display import display, Image as IPImage
from pathlib import Path

# Show comparison sheet
comparison = run_path / 'samples' / 'comparison.png'
if comparison.exists():
    print('Baseline (left) vs LoRA (right):')
    display(IPImage(filename=str(comparison), width=800))
else:
    print('No comparison sheet generated')

In [ ]:
# Show individual samples side by side
from IPython.display import display, Image as IPImage
from PIL import Image
import matplotlib.pyplot as plt

baseline_dir = run_path / 'samples' / 'baseline'
final_dir = run_path / 'samples' / 'final'

if baseline_dir.exists() and final_dir.exists():
    baseline_imgs = sorted(baseline_dir.glob('*.png'))[:6]
    final_imgs = sorted(final_dir.glob('*.png'))[:6]
    
    n = min(len(baseline_imgs), len(final_imgs))
    if n > 0:
        fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
        if n == 1:
            axes = axes.reshape(2, 1)
        for i in range(n):
            axes[0, i].imshow(Image.open(baseline_imgs[i]))
            axes[0, i].set_title(f'Baseline: {baseline_imgs[i].name}', fontsize=8)
            axes[0, i].axis('off')
            axes[1, i].imshow(Image.open(final_imgs[i]))
            axes[1, i].set_title(f'LoRA: {final_imgs[i].name}', fontsize=8)
            axes[1, i].axis('off')
        plt.suptitle('Baseline vs LoRA Samples', fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print(f'Sample dirs not found: baseline={baseline_dir.exists()}, final={final_dir.exists()}')

## 6. Download Results

In [ ]:
import shutil

# Archive the run
archive_path = f'/content/{run_path.name}.zip'
shutil.make_archive(archive_path.replace('.zip', ''), 'zip', root_dir=run_path.parent, base_dir=run_path.name)
print(f'Archive: {archive_path}')
print(f'Size: {Path(archive_path).stat().st_size / 1024 / 1024:.1f} MB')

# Download (Colab)
# from google.colab import files
# files.download(archive_path)